In [1]:
!pip install transformers accelerate torch

Load the Model into GPU Memory (Updated)

In [2]:
import torch
from transformers import pipeline

print("⏳ Loading a fully un-gated open-source model into Colab memory...")

# Initializing Qwen 2.5 1.5B Instruct - fits beautifully and instantly on a T4 GPU
pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto"
)
print("✅ Model loaded successfully on the GPU with zero authentication required!")

⏳ Loading a fully un-gated open-source model into Colab memory...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ Model loaded successfully on the GPU with zero authentication required!


Define the Evaluation Function

In [5]:
import json
import re

def interview_turn_local(job_title, job_desc, conversation_history, user_answer=None):
    """
    If user_answer is None, it's the start of the interview; generate question 1.
    Otherwise, evaluate user_answer and generate the next question.
    """

    # Precise instructions forcing the model to return both feedback AND the next question in JSON
    system_instruction = (
        "You are an expert cybersecurity and technical interviewer conducting a live interactive interview.\n"
        "Analyze the candidate's response against the job title and description.\n"
        "You must respond ONLY with a valid JSON object matching this exact schema:\n"
        "{\n"
        '  "score_content": "1-10 string or N/A if first question",\n'
        '  "score_delivery": "1-10 string or N/A if first question",\n'
        '  "feedback_strengths": "Brief statement of what they did well, or N/A if first question",\n'
        '  "feedback_weaknesses": "Brief constructive criticism, or N/A if first question",\n'
        '  "next_question": "Your next highly relevant technical or behavioral interview question"\n'
        "}"
    )

    # Set up the context prompt
    context_prompt = f"--- JOB CONTEXT ---\nJob Title: {job_title}\nKey Skills: {job_desc}\n\n"

    if user_answer is None:
        # Initial call to get the very first question
        user_content = f"{context_prompt}Start the interview. Generate the first technical or behavioral question."
    else:
        # Subsequent calls: pass the history context, the last question, and the user's answer
        user_content = f"{context_prompt}"
        if conversation_history:
            user_content += f"Previous Question Asked: {conversation_history[-1]['question']}\n"
        user_content += f"Candidate Answer: {user_answer}\n\nEvaluate this answer and provide the 'next_question'."

    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": user_content}
    ]

    outputs = pipe(
        messages,
        max_new_tokens=600,
        do_sample=True,
        temperature=0.2, # Lower temperature enforces strict adherence to JSON layout
        top_p=0.9
    )

    raw_text = outputs[0]["generated_text"][-1]["content"].strip()

    try:
        # Extract json text specifically if the model added markdown blocks or extra characters
        json_match = re.search(r'\{.*\}', raw_text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group(0))
        return json.loads(raw_text)
    except Exception:
        # Graceful fallback text if parsing hiccups occur
        return {
            "score_content": "Completed",
            "score_delivery": "Completed",
            "feedback_strengths": "Answer recorded.",
            "feedback_weaknesses": "Could not parse precise metrics natively.",
            "next_question": "Can you elaborate on your experience with core security frameworks or protocols?"
        }

In [8]:
def run_dynamic_interview():
    print("="*60)
    print("🤖 ACTIVE AI INTERVIEW COACH (5-QUESTION CHALLENGE)")
    print("="*60)

    job_title = input("\nEnter Target Job Title (e.g., Security Analyst): ")
    job_desc = input("Paste Job Description / Key Skills (e.g., nmap, wireshark): ")

    print("\n⏳ Initializing the interview engine and generating Question #1...")

    conversation_history = []
    MAX_QUESTIONS = 5

    # Get the very first question
    turn_data = interview_turn_local(job_title, job_desc, conversation_history, user_answer=None)
    current_question = turn_data.get("next_question", "Tell me about your experience in this field?")

    question_count = 1

    while question_count <= MAX_QUESTIONS:
        print("\n" + "="*50)
        print(f"🤖 INTERVIEWER QUESTION #{question_count} of {MAX_QUESTIONS}:")
        print(f"👉 {current_question}")
        print("="*50)

        user_answer = input("\n🗣️ Your Answer (or type 'exit' to quit): ")
        if user_answer.strip().lower() == 'exit':
            print("\n👋 Interview session terminated early. Good luck practicing!")
            return

        print("\n⏳ Interviewer is evaluating your response...")

        conversation_history.append({"question": current_question})

        # Get evaluation of the answer and fetch the next question
        turn_data = interview_turn_local(job_title, job_desc, conversation_history, user_answer=user_answer)

        # Display the validation dashboard for the answer just given
        print("\n📊 --- ANSWER #{count} VALIDATION ---".format(count=question_count))
        print(f"🔹 Content Accuracy : {turn_data.get('score_content')}/10")
        print(f"🔹 Delivery Metric  : {turn_data.get('score_delivery')}/10")
        print(f"✅ What went well   : {turn_data.get('feedback_strengths')}")
        print(f"⚠️ Where to improve : {turn_data.get('feedback_weaknesses')}")

        # Check if we just completed the final question
        if question_count == MAX_QUESTIONS:
            print("\n" + "="*60)
            print("🎉 INTERVIEW COMPLETE! You have answered all 5 questions.")
            print("🏆 Review your final feedback cards above to refine your prep.")
            print("="*60)
            break

        # Advance to the next question if we haven't hit the limit yet
        current_question = turn_data.get("next_question", "Can you walk me through your standard approach to troubleshooting?")
        question_count += 1

In [9]:
if __name__ == "__main__":
    run_dynamic_interview()

🤖 ACTIVE AI INTERVIEW COACH (5-QUESTION CHALLENGE)

Enter Target Job Title (e.g., Security Analyst): cyber security analyst
Paste Job Description / Key Skills (e.g., nmap, wireshark): nmap


[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



⏳ Initializing the interview engine and generating Question #1...

🤖 INTERVIEWER QUESTION #1 of 5:
👉 Can you walk us through how you would use nmap to scan for open ports on a target network?

🗣️ Your Answer (or type 'exit' to quit): To scan for open ports on a target network with Nmap, you need to first identify active hosts, then probe those devices for open, closed, or filtered ports, and finally analyze the active services to assess your network's attack surface


[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



⏳ Interviewer is evaluating your response...

📊 --- ANSWER #1 VALIDATION ---
🔹 Content Accuracy : 8/10
🔹 Delivery Metric  : 7/10
✅ What went well   : The candidate provided a clear explanation of using Nmap for port scanning. They mentioned identifying active hosts, probing for open ports, and analyzing services, which is accurate.
⚠️ Where to improve : N/A

🤖 INTERVIEWER QUESTION #2 of 5:
👉 Can you demonstrate how you would perform a stealthy scan using Nmap? This will help us understand your understanding of ethical hacking practices.

🗣️ Your Answer (or type 'exit' to quit): To run a SYN stealth scan, you must have root or sudo privileges to generate raw packets.A standard stealth scan syntax is:sudo nmap -sS target_ip


...


⏳ Interviewer is evaluating your response...

📊 --- ANSWER #3 VALIDATION ---
🔹 Content Accuracy : 8/10
🔹 Delivery Metric  : Good explanation showing understanding of the concept./10
✅ What went well   : The candidate correctly identified that root/sudo access is necessary for creating raw sockets in a stealthy manner. This demonstrates knowledge of the underlying principles involved in such scans.
⚠️ Where to improve : N/A

🤖 INTERVIEWER QUESTION #4 of 5:
👉 Can you walk us through how a stealthy scan using nmap can be performed without compromising system integrity?

🗣️ Your Answer (or type 'exit' to quit): An Nmap stealth scan—commonly known as a SYN scan or Half-Open scan (-sS)—gathers port information without completing the full three-way TCP handshake, thereby minimizing detection and system resource impact.


[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



⏳ Interviewer is evaluating your response...

📊 --- ANSWER #4 VALIDATION ---
🔹 Content Accuracy : 8/10
🔹 Delivery Metric  : Good explanation but could include more details on why stealth is important in cybersecurity./10
✅ What went well   : The candidate provided a clear explanation of the SYN scan technique used by Nmap for stealthy scanning. They also mentioned that it minimizes detection and resource impact, which is accurate.
⚠️ Where to improve : To improve, the candidate should elaborate further on why stealth is crucial in cybersecurity, especially when discussing potential vulnerabilities or threats.

🤖 INTERVIEWER QUESTION #5 of 5:
👉 Can you explain how understanding network protocols like TCP/IP helps in identifying potential vulnerabilities during a stealthy scan?

🗣️ Your Answer (or type 'exit' to quit): Understanding TCP/IP allows you to decipher how systems respond to network probes without completing a full connection


[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



⏳ Interviewer is evaluating your response...

📊 --- ANSWER #5 VALIDATION ---
🔹 Content Accuracy : 8/10
🔹 Delivery Metric  : Good explanation but could be more detailed./10
✅ What went well   : The candidate provided a clear explanation that aligns with the need for understanding network protocols to identify vulnerabilities. However, it would have been helpful if the candidate elaborated on why knowledge of TCP/IP is crucial for stealthy scans.
⚠️ Where to improve : The candidate should have expanded on the specific ways in which TCP/IP protocol details can aid in detecting vulnerabilities during stealthy scans.

🎉 INTERVIEW COMPLETE! You have answered all 5 questions.
🏆 Review your final feedback cards above to refine your prep.
